# Predicción de GHI (Irradiancia Horizontal Global) con BiLSTM

**Demo producto — Valle de Aburrá · Climate Week 2026 · Emergente**

| Paso | Título | Descripción |
|------|--------|-------------|
| 1 | Configuración | Librerías, rutas y paleta de colores |
| 2 | Datos de entrada | La ventana de 37 pasos y 79 variables que el modelo recibe |
| 3 | Arquitectura del modelo | Cómo BiLSTM procesa la secuencia temporal |
| 4 | Las 13 predicciones horarias | Tabla CSI × cielo despejado = GHI (W/m²) |
| 5 | Comparación con SIATA | Animación: predicción vs. medición real |
| 6 | Comparación con GFS crudo | Animación: BiLSTM vs. GFS vs. SIATA |
| 7 | Producto para el operador | Pronóstico entregado a las 4:00 AM |

---

**Fecha objetivo:** 15 de septiembre de 2021  
**Ubicación:** Valle de Aburrá, Medellín (6.25°N, 75.5°W, 1485 m s.n.m.)

In [ ]:
# ── Install required packages silently ───────────────────────────
import subprocess, sys

_PACKAGES = ['xarray', 'h5netcdf', 'pvlib', 'torch', 'pandas', 'numpy', 'matplotlib']
for _pkg in _PACKAGES:
    try:
        __import__(_pkg)
    except ImportError:
        print(f'Instalando {_pkg}...')
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', _pkg],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ── Standard imports ──────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display
import torch
import xarray as xr
warnings.filterwarnings('ignore')

# ── Project root detection (walk up from cwd) ─────────────────────
_search = os.path.abspath(os.getcwd())
PROJECT_ROOT = None
for _ in range(6):
    if os.path.isdir(os.path.join(_search, '_4_LSTM_modules')):
        PROJECT_ROOT = _search
        break
    _search = os.path.dirname(_search)
if PROJECT_ROOT is None:
    PROJECT_ROOT = r'C:\Users\isabe\Projects\codigors\carpetasdetrabajo'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'PROJECT_ROOT → {PROJECT_ROOT}')

# ── File paths ────────────────────────────────────────────────────
NPZ_PATH = os.path.join(PROJECT_ROOT,
    '_4_LSTM_modules', 'Prepared_data',
    '4launch_multfeat_sym18_clim.npz')

MODEL_PATH = os.path.join(PROJECT_ROOT,
    '_4_LSTM_modules', '_runs', '4launch_multfeat_sym',
    '4launch_Multfeat_sym18_clim79_FIXED_BiLSTM_attn_20260706_102943',
    'best_model.pt')

PRED_REAL_PATH = os.path.join(PROJECT_ROOT,
    '_4_LSTM_modules', '_runs', '4launch_multfeat_sym',
    '4launch_Multfeat_sym18_clim79_FIXED_BiLSTM_attn_20260706_102943',
    'pred_real.csv')

CSI_GHI_FILE = os.path.join(PROJECT_ROOT,
    '_3_Data_preparation_for_LSTM', 'Preparation_data',
    '_01_CSI_EXT_radiation', 'Ineichen_GHI',
    'CSI_GHI_grid25_avg_with_horizon_and_enhancement_with_bias_correct2.nc')

SIATA_GHI_FILE = os.path.join(PROJECT_ROOT,
    '_3_Data_preparation_for_LSTM', 'Preparation_data',
    '_03_Siata_GHI', 'Netcdf_Siata_GHI', 'SIATA_GHI_all.nc')

GFS_CSI_FILE = os.path.join(PROJECT_ROOT,
    '_3_Data_preparation_for_LSTM', 'Preparation_data',
    '_04_indices', 'clear_sky_indices', 'clearsky_index_GFS_0700.nc')

TARGET_DATE = '2021-09-15'
TARGET_DT   = pd.Timestamp('2021-09-15 12:00:00')

# ── Color palette ─────────────────────────────────────────────────
COLOR_BILSTM = '#2E86AB'
COLOR_SIATA  = '#1B4332'
COLOR_GFS    = '#E07A5F'
COLOR_CS     = '#A8DADC'
COLOR_PAST   = '#F0F0F0'
COLOR_CENTER = '#FFB703'
COLOR_FUTURE = '#E8F4FD'

# ── Clean matplotlib style ────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.color':        '#E8E8E8',
    'grid.linewidth':    0.8,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'legend.fontsize':   10,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'figure.dpi':        120,
})
# Use pillow writer for animations (avoids ffmpeg dependency)
plt.rcParams['animation.writer'] = 'pillow'

print('✓ Entorno configurado correctamente')
print(f'  NPZ      : {os.path.basename(NPZ_PATH)}')
print(f'  Modelo   : {os.path.basename(os.path.dirname(MODEL_PATH))}')

## Paso 2: Los datos que entran al modelo

El modelo genera **13 predicciones independientes**, una por cada hora diurna (6:00 AM a 6:00 PM).

Para predecir cada hora, el modelo recibe una **ventana simétrica de 37 pasos temporales**:

| Segmento | Pasos | Descripción |
|----------|-------|-------------|
| Pasado | t-18 a t-1 (18 pasos) | Pronóstico GFS ya disponible para horas pasadas |
| **Hora central** | **t=0** | **La hora que se quiere predecir** |
| Futuro | t+1 a t+18 (18 pasos) | Pronóstico GFS para las horas siguientes |

**Por cada paso temporal: 79 variables del GFS** (radiación, nubosidad, temperatura, humedad, viento...)

> **Total:** 37 pasos × 79 variables = **2.923 valores de entrada** para predecir **1 hora**

### Ejemplo: predecir las 12:00pm del 15 de septiembre 2021
- Ventana inicia: **Sep 14, 18:00** (18 horas antes)
- Hora central: **Sep 15, 12:00** (la hora a predecir)
- Ventana termina: **Sep 16, 06:00** (18 horas después)

In [ ]:
# ── Graph 1: 13 daytime hours the model predicts ─────────────────
fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Night shading (hours 0–5 and 19–23)
ax.axvspan(-0.5, 5.5,  alpha=0.12, color='#555555', zorder=0)
ax.axvspan(18.5, 23.5, alpha=0.12, color='#555555', zorder=0)

ax.text(2.5,  0.85, 'Noche', ha='center', va='center', fontsize=11,
        color='#777777', transform=ax.get_xaxis_transform())
ax.text(21.0, 0.85, 'Noche', ha='center', va='center', fontsize=11,
        color='#777777', transform=ax.get_xaxis_transform())

# Orange circles and upward arrows for the 13 prediction hours
for h in range(6, 19):
    ax.annotate('', xy=(h, 0.62), xytext=(h, 0.75),
                xycoords=('data', 'axes fraction'),
                textcoords=('data', 'axes fraction'),
                arrowprops=dict(arrowstyle='->', color=COLOR_CENTER, lw=1.8))
    circle = plt.Circle((h, 0), 0.35, color=COLOR_CENTER, zorder=5)
    ax.add_patch(circle)
    ax.text(h, -0.55, f'{h}:00', ha='center', va='top', fontsize=9, color='#333333')

# Span label "13 predicciones"
ax.annotate('', xy=(18, 0.90), xytext=(6, 0.90),
            xycoords=('data', 'axes fraction'),
            textcoords=('data', 'axes fraction'),
            arrowprops=dict(arrowstyle='<->', color='#333333', lw=1.5))
ax.text(12, 0.96, '13 predicciones independientes',
        ha='center', va='bottom', fontsize=12, fontweight='bold',
        color='#333333', transform=ax.get_xaxis_transform())

ax.set_xlim(-0.5, 23.5)
ax.set_ylim(-1.2, 1.5)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.grid(False)

ax.set_title('Las 13 horas diurnas que predice el modelo',
             fontsize=14, fontweight='bold', pad=14)
ax.text(0.5, -0.12, 'Una predicción independiente por hora · cada una usa una ventana de 37 pasos',
        ha='center', va='top', transform=ax.transAxes,
        fontsize=10, color='#666666', style='italic')

plt.tight_layout()
plt.savefig(os.path.join(os.getcwd(), 'graph1_13hours.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Graph 2: symmetric 37-step window centred at 2021-09-15 12:00 ─
fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.set_xlim(-1, 38)
ax.set_ylim(-2.5, 4.5)
ax.axis('off')

center_ts  = pd.Timestamp('2021-09-15 12:00')
timestamps = [center_ts + pd.Timedelta(hours=i - 18) for i in range(37)]
box_w, box_h = 0.82, 0.85

for i, ts in enumerate(timestamps):
    x = i * (box_w + 0.08)
    if i < 18:
        fc, ec = COLOR_PAST,   '#CCCCCC'
    elif i == 18:
        fc, ec = COLOR_CENTER, '#CC8800'
    else:
        fc, ec = COLOR_FUTURE, '#85C1E9'

    rect = FancyBboxPatch((x, 0), box_w, box_h, boxstyle='round,pad=0.03',
                          facecolor=fc, edgecolor=ec, linewidth=1.2, zorder=3)
    ax.add_patch(rect)

    # Step label above (every 6 steps + center)
    if i == 18:
        ax.text(x + box_w/2, box_h + 0.18, 't=0', ha='center', va='bottom',
                fontsize=9, fontweight='bold', color='#CC8800')
    elif i % 6 == 0 or i == 36:
        ax.text(x + box_w/2, box_h + 0.15, f't{i - 18:+d}', ha='center', va='bottom',
                fontsize=8, color='#555555')

    # Timestamp below (every 6 steps + center)
    if i in (0, 6, 12, 18, 24, 30, 36):
        day = ts.day
        if day == 14:
            lbl = ts.strftime('%H:%M\nSep 14')
        elif day == 15:
            lbl = ts.strftime('%H:%M\nSep 15')
        else:
            lbl = ts.strftime('%H:%M\nSep 16')
        ax.text(x + box_w/2, -0.22, lbl, ha='center', va='top', fontsize=8,
                color='#333333' if i == 18 else '#666666',
                fontweight='bold' if i == 18 else 'normal')

# Arrow above center box
center_x = 18 * (box_w + 0.08) + box_w / 2
ax.annotate('CSI predicho', xy=(center_x, box_h + 0.05), xytext=(center_x, box_h + 1.6),
            ha='center', fontsize=11, fontweight='bold', color=COLOR_CENTER,
            arrowprops=dict(arrowstyle='->', color=COLOR_CENTER, lw=2.0),
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec=COLOR_CENTER, alpha=0.9, lw=1.5))

legend_handles = [
    mpatches.Patch(facecolor=COLOR_PAST,   edgecolor='#CCCCCC', label='Pasado (18 pasos)'),
    mpatches.Patch(facecolor=COLOR_CENTER, edgecolor='#CC8800', label='Hora central t=0  (12:00pm Sep 15)'),
    mpatches.Patch(facecolor=COLOR_FUTURE, edgecolor='#85C1E9', label='Futuro GFS (18 pasos)'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=9,
          framealpha=0.92, bbox_to_anchor=(0.0, 1.05))

ax.set_title('Ventana de 37 pasos — predicción 12:00pm del 15 de septiembre 2021',
             fontsize=13, fontweight='bold', pad=30)
ax.text(0.5, -0.18, 'El GFS ya tiene pronóstico disponible para todas las horas futuras de la ventana',
        ha='center', va='top', transform=ax.transAxes,
        fontsize=10, color='#666666', style='italic')

plt.tight_layout()
plt.savefig(os.path.join(os.getcwd(), 'graph2_window.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Load NPZ and extract sequence for 2021-09-15 12:00 ───────────
try:
    data = np.load(NPZ_PATH, allow_pickle=True)
    feature_vars = list(data['feature_vars'])
    print(f'NPZ cargado: {os.path.basename(NPZ_PATH)}')
    print(f'  X_train: {data["X_train"].shape}  |  features: {len(feature_vars)}')
except FileNotFoundError:
    print(f'ERROR: No se encontró el archivo NPZ:\n  {NPZ_PATH}')
    raise

# Merge all splits (used in Cell 9 for full-day inference)
X_all = np.concatenate([data['X_train'], data['X_val'], data['X_test']])
y_all = np.concatenate([data['y_train'], data['y_val'], data['y_test']])
t_all = pd.to_datetime(
    np.concatenate([data['t_train'], data['t_val'], data['t_test']]))

# Find 2021-09-15 12:00 sequence
mask_target = (t_all == TARGET_DT)
if not mask_target.any():
    raise ValueError(f'No se encontró la secuencia para {TARGET_DT}')
idx   = int(np.where(mask_target)[0][0])
X_seq = X_all[idx]    # (37, 79)
print(f'  Secuencia {TARGET_DT}: shape {X_seq.shape}')

# ── Select 5 variables (flexible lookup) ─────────────────────────
VARS_5 = ['dswrf1_0700', 'TCDC_ent_0700', 'CAPE_surface_0700', 'RH_2m_0700', 'zenith']
avail_5     = [v for v in VARS_5 if v in feature_vars]
col_indices = [feature_vars.index(v) for v in avail_5]
print(f'  Variables seleccionadas: {avail_5}')

# Normalize each variable to [0,1] for display
X_5    = X_seq[:, col_indices]
X_norm = np.zeros_like(X_5, dtype=float)
for j in range(X_5.shape[1]):
    vmin, vmax = X_5[:, j].min(), X_5[:, j].max()
    rng = vmax - vmin
    X_norm[:, j] = (X_5[:, j] - vmin) / rng if rng > 0 else 0.5

# Build x-axis labels
center_ts    = pd.Timestamp('2021-09-15 12:00')
win_times    = [center_ts + pd.Timedelta(hours=i - 18) for i in range(37)]
x_steps      = list(range(37))
xtick_pos    = [0, 6, 12, 18, 24, 30, 36]
xtick_labels = []
for p in xtick_pos:
    ts = win_times[p]
    d  = ts.day
    if d == 14:
        xtick_labels.append(ts.strftime('%H:%M\nSep 14'))
    elif d == 15:
        xtick_labels.append(ts.strftime('%H:%M\nSep 15'))
    else:
        xtick_labels.append(ts.strftime('%H:%M\nSep 16'))

# ── Line plot ─────────────────────────────────────────────────────
COLORS_5 = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
fig, ax = plt.subplots(figsize=(14, 6))

for j, (var_name, color) in enumerate(zip(avail_5, COLORS_5)):
    ax.plot(x_steps, X_norm[:, j], color=color, linewidth=2.0, label=var_name, zorder=4)

# Center shading
ax.axvspan(17.5, 18.5, alpha=0.25, color=COLOR_CENTER, zorder=2)
ax.text(18, 1.03, '12:00pm\nHora a predecir', ha='center', va='bottom',
        fontsize=9, fontweight='bold', color='#CC8800',
        transform=ax.get_xaxis_transform())

ax.set_xticks(xtick_pos)
ax.set_xticklabels(xtick_labels, fontsize=9)
ax.set_ylabel('Valor normalizado [0–1]', fontsize=11)
ax.set_ylim(-0.05, 1.20)
ax.set_title('5 variables del GFS a lo largo de la ventana de 37 pasos',
             fontsize=13, fontweight='bold')
ax.text(0.5, 1.01,
        '79 variables en total  ·  37 × 79 = 2.923 valores de entrada para predecir 1 hora',
        ha='center', va='bottom', transform=ax.transAxes,
        fontsize=9.5, color='#555555', style='italic')
ax.legend(loc='upper right', bbox_to_anchor=(1.18, 1.0), fontsize=10, framealpha=0.92)

plt.tight_layout()
plt.savefig(os.path.join(os.getcwd(), 'graph3_5vars.png'), dpi=120, bbox_inches='tight')
plt.show()

## Paso 3: ¿Cómo procesa el modelo la información?

El modelo BiLSTM transforma los 2.923 valores de entrada en **un solo número: el CSI predicho** (Índice de Cielo Despejado, entre 0 y 1).

El procesamiento ocurre en 5 etapas secuenciales:

| Bloque | Función | Detalle técnico |
|--------|---------|------------------|
| **1. Entrada** | Recibe la secuencia | 37 pasos × 79 variables = 2.923 valores |
| **2. LayerNorm** | Normaliza las variables | Lleva todas las features a la misma escala antes del LSTM |
| **3. BiLSTM × 3 capas** | Aprende patrones temporales | Lee la secuencia hacia adelante Y hacia atrás simultáneamente (96 neuronas/dirección) |
| **4. Dropout 25%** | Regularización | Apaga neuronas al azar durante el entrenamiento para evitar memorización |
| **5. Sigmoid** | Restricción física | Fuerza la salida entre 0 y 1 — el CSI es un ratio, no puede ser negativo ni superar 1 |

**Parámetros totales del modelo: 581,920**

In [ ]:
# ── Architecture block diagram: 5 blocks + output ────────────────
fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.set_xlim(-0.5, 18.5)
ax.set_ylim(-1.5, 4.0)
ax.axis('off')

blocks = [
    {'x': 0.0,  'fc': '#E8E8E8', 'ec': '#AAAAAA', 'title_color': '#333333',
     'title': 'Entrada',
     'text': '37 pasos × 79 variables\n2.923 valores'},
    {'x': 3.2,  'fc': '#AED6F1', 'ec': '#AAAAAA', 'title_color': '#1A5276',
     'title': 'LayerNorm',
     'text': 'Normaliza\ntodas las variables\na la misma escala'},
    {'x': 6.4,  'fc': '#2E86AB', 'ec': 'white',   'title_color': 'white',
     'title': 'BiLSTM × 3 capas',
     'text': '96 neuronas / dirección\nAprende microclima\ndel valle'},
    {'x': 9.6,  'fc': '#F0A500', 'ec': 'white',   'title_color': '#7B3F00',
     'title': 'Dropout 25%',
     'text': 'Apaga neuronas\nal azar\nPreviene memorización'},
    {'x': 12.8, 'fc': '#1B4332', 'ec': 'white',   'title_color': 'white',
     'title': 'Sigmoid',
     'text': 'Fuerza salida\nentre 0 y 1\nRestr. física CSI'},
]

box_w, box_h = 2.8, 2.0
for i, blk in enumerate(blocks):
    x = blk['x']
    rect = FancyBboxPatch((x, 0), box_w, box_h, boxstyle='round,pad=0.1',
                          facecolor=blk['fc'], edgecolor=blk['ec'],
                          linewidth=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(x + box_w/2, box_h - 0.22, blk['title'],
            ha='center', va='top', fontsize=10.5, fontweight='bold',
            color=blk['title_color'], zorder=4)
    ax.text(x + box_w/2, box_h/2 - 0.2, blk['text'],
            ha='center', va='center', fontsize=8.5,
            color=blk['title_color'], zorder=4, linespacing=1.6)
    if i < len(blocks) - 1:
        ax.annotate('', xy=(x + box_w + 0.32, box_h/2),
                    xytext=(x + box_w + 0.02, box_h/2),
                    arrowprops=dict(arrowstyle='->', color='#555555', lw=2.0))

# Output box
out_x = 16.2
out_rect = FancyBboxPatch((out_x, 0.3), 2.0, 1.4, boxstyle='round,pad=0.15',
                          facecolor=COLOR_CENTER, edgecolor='#CC8800',
                          linewidth=2, zorder=3)
ax.add_patch(out_rect)
ax.text(out_x + 1.0, 1.0, 'CSI predicho\npara las\n12:00pm',
        ha='center', va='center', fontsize=9.5, fontweight='bold',
        color='#7B3F00', zorder=4)
ax.annotate('', xy=(out_x - 0.08, box_h/2),
            xytext=(blocks[-1]['x'] + box_w + 0.05, box_h/2),
            arrowprops=dict(arrowstyle='->', color='#555555', lw=2.0))

ax.set_title('Arquitectura del modelo — de entrada a predicción',
             fontsize=14, fontweight='bold', y=0.98)

plt.tight_layout()
plt.savefig(os.path.join(os.getcwd(), 'graph7_architecture.png'), dpi=120, bbox_inches='tight')
plt.show()

## Paso 4: Las 13 predicciones horarias

El proceso de la ventana de 37 pasos se repite **13 veces**, una para cada hora diurna.

Cada predicción produce un **CSI** (Índice de Cielo Despejado) entre 0 y 1:
- CSI ≈ 1.0 → cielo completamente despejado
- CSI ≈ 0.5 → mitad de la radiación esperada (nubes parciales)
- CSI ≈ 0.0 → cielo completamente cubierto

Para convertir a W/m²:

$$\text{GHI predicho (W/m²)} = \text{CSI predicho} \times \text{GHI cielo despejado (Ineichen)}$$

La siguiente tabla muestra las 13 predicciones para el 15 de septiembre de 2021:

In [ ]:
# ── Load BiLSTM model ─────────────────────────────────────────────
try:
    from _4_LSTM_modules.NN_modules.BiLSTMRegressor import BiLSTMRegressor
    model_bilstm = BiLSTMRegressor(
        n_feat=79, hidden=96, num_layers=3, dropout=0.25, activation='sigmoid')
    state = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
    model_bilstm.load_state_dict(state)
    model_bilstm.eval()
    n_params = sum(p.numel() for p in model_bilstm.parameters())
    print(f'Modelo cargado: {n_params:,} parámetros')
except FileNotFoundError:
    print(f'ERROR: Modelo no encontrado:\n  {MODEL_PATH}')
    raise

# ── Find all sequences for 2021-09-15 (06–18h) ───────────────────
mask_day = ((t_all >= '2021-09-15 06:00') & (t_all <= '2021-09-15 18:00'))
order    = np.argsort(t_all[mask_day])
X_day    = X_all[mask_day][order]
t_day    = t_all[mask_day][order]
print(f'Secuencias diurnas encontradas: {len(t_day)} de 13')

# ── Run inference ─────────────────────────────────────────────────
with torch.no_grad():
    csi_pred_day = model_bilstm(torch.tensor(X_day, dtype=torch.float32)).numpy()
print(f'CSI predicho — rango: [{csi_pred_day.min():.3f}, {csi_pred_day.max():.3f}]')

# ── Load clear-sky GHI ────────────────────────────────────────────
try:
    ds_cs = xr.open_dataset(CSI_GHI_FILE, engine='h5netcdf')
    cs_series = pd.Series(
        ds_cs['clear_sky_ghi'].values.squeeze(),
        index=pd.to_datetime(ds_cs['observation_time'].values))
    ds_cs.close()
    cs_values_day = cs_series.reindex(t_day).values
    print(f'Cielo despejado — rango: [{cs_values_day.min():.0f}, {cs_values_day.max():.0f}] W/m²')
except FileNotFoundError:
    print(f'ERROR: No se encontró el archivo de cielo despejado:\n  {CSI_GHI_FILE}')
    raise

ghi_pred_day = csi_pred_day * cs_values_day   # W/m²

# Store for later cells
csi_pred_all  = csi_pred_day.copy()
ghi_pred_all  = ghi_pred_day.copy()
t_day_all     = t_day.copy()
cs_values_all = cs_values_day.copy()

# Summary
avg_ghi_day = ghi_pred_day[ghi_pred_day > 0].mean()
peak_idx    = int(np.argmax(ghi_pred_day))
peak_hour   = t_day[peak_idx].strftime('%H:%M')
peak_ghi    = ghi_pred_day[peak_idx]
print(f'GHI promedio diurno : {avg_ghi_day:.0f} W/m²')
print(f'Máxima predicha     : {peak_ghi:.0f} W/m² a las {peak_hour}')

# ── Styled DataFrame ──────────────────────────────────────────────
df_table = pd.DataFrame({
    'Hora':                       [ts.strftime('%H:%M') for ts in t_day],
    'CSI predicho':               np.round(csi_pred_day, 3),
    'GHI cielo despejado (W/m²)': np.round(cs_values_day, 0).astype(int),
    'GHI predicho (W/m²)':        np.round(ghi_pred_day, 0).astype(int),
}).reset_index(drop=True)

_cm_orange = LinearSegmentedColormap.from_list('wh_orange', ['#FFFFFF', '#E07A5F'], N=256)
_peak_cs   = float(cs_values_day.max()) * 1.05

def _highlight_peak_row(row):
    if int(row['GHI predicho (W/m²)']) == int(round(peak_ghi)):
        return ['font-weight: bold; background-color: #FFF3CD'] * len(row)
    return [''] * len(row)

styled_table = (
    df_table.style
    .background_gradient(cmap=_cm_orange, subset=['GHI predicho (W/m²)'],
                         vmin=0, vmax=_peak_cs)
    .apply(_highlight_peak_row, axis=1)
    .format({'CSI predicho': '{:.3f}',
             'GHI cielo despejado (W/m²)': '{:.0f}',
             'GHI predicho (W/m²)': '{:.0f}'})
    .set_caption(
        f'Predicciones BiLSTM — 15 de septiembre 2021  ·  '
        f'Promedio diurno: {avg_ghi_day:.0f} W/m²  ·  '
        f'GHI = CSI predicho × GHI cielo despejado')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '12px'), ('font-weight', 'bold'), ('color', '#333')]},
        {'selector': 'th',
         'props': [('background-color', COLOR_BILSTM),
                   ('color', 'white'), ('font-size', '11px'), ('padding', '6px 14px')]},
        {'selector': 'td',
         'props': [('font-size', '11px'), ('padding', '5px 14px')]},
    ])
)
display(styled_table)

## Paso 5: Comparación con la medición real SIATA

Hasta ahora el modelo **no sabía** qué ocurrió realmente ese día.

Ahora revelamos qué midió la red **SIATA** (Sistema de Alerta Temprana del Valle de Aburrá).

> La SIATA opera una red de sensores meteorológicos distribuidos en el Valle de Aburrá.  
> Su medición de GHI representa la **verdad de campo** para evaluar el modelo.

La animación muestra primero la predicción del modelo, y luego la medición real:

In [ ]:
# ── Hourly grid and load SIATA GHI for 2021-09-15 ────────────────
HOURS_IDX = pd.date_range('2021-09-15 06:00', '2021-09-15 18:00', freq='1h')

def _align_hourly(timestamps, values):
    """Align arbitrary time series to the daytime hourly grid."""
    s = pd.Series(values, index=pd.to_datetime(timestamps))
    s = s[~s.index.duplicated(keep='first')]
    return s.reindex(HOURS_IDX).values

try:
    ds_s    = xr.open_dataset(SIATA_GHI_FILE, engine='h5netcdf')
    t_s_all = pd.to_datetime(ds_s['observation_time'].values)
    v_s_all = ds_s['GHI'].values.squeeze()
    ds_s.close()
    _m = (t_s_all >= TARGET_DATE) & (t_s_all < '2021-09-16')
    siata_vals = _align_hourly(t_s_all[_m], v_s_all[_m])
    print(f'SIATA cargado: {_m.sum()} registros  ·  {int((siata_vals > 0).sum())} horas con GHI > 0')
except FileNotFoundError:
    print(f'ERROR: No se encontró SIATA_GHI_all.nc:\n  {SIATA_GHI_FILE}')
    raise

# Align BiLSTM predictions to the hourly grid
bilstm_vals = (pd.Series(ghi_pred_all, index=t_day_all)
               .reindex(HOURS_IDX).fillna(0).values)
siata_vals  = np.nan_to_num(siata_vals, nan=0.0)

x_hours = list(range(len(HOURS_IDX)))
xlabels = [ts.strftime('%H:%M') for ts in HOURS_IDX]
n_pts   = len(x_hours)

# Compute RMSE for this day
_m_rmse  = (siata_vals > 0) & (bilstm_vals > 0)
rmse_day = np.sqrt(np.mean((bilstm_vals[_m_rmse] - siata_vals[_m_rmse]) ** 2))
print(f'RMSE BiLSTM vs SIATA el 15 sep 2021: {rmse_day:.1f} W/m²')

# ── FuncAnimation: 3 phases ───────────────────────────────────────
#   Phase 1 (frames 0–25):  BiLSTM line draws left to right
#   Phase 2 (frames 26–51): SIATA line draws over it
#   Phase 3 (frames 52–64): both visible + metrics box
TOTAL_FRAMES1 = 65
y_max1 = max(siata_vals.max(), bilstm_vals.max()) * 1.18

fig_a1, ax_a1 = plt.subplots(figsize=(12, 6))
ax_a1.set_xlim(-0.5, n_pts - 0.5)
ax_a1.set_ylim(0, y_max1)
ax_a1.set_xticks(x_hours)
ax_a1.set_xticklabels(xlabels, rotation=30, ha='right')
ax_a1.set_xlabel('Hora local (Colombia, UTC-5)', fontsize=11)
ax_a1.set_ylabel('GHI (W/m²)', fontsize=11)
ax_a1.set_title('15 de septiembre 2021 — Valle de Aburrá', fontsize=14, fontweight='bold')
ax_a1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

line_b1, = ax_a1.plot([], [], color=COLOR_BILSTM, lw=2.8, label='Predicción BiLSTM', zorder=4)
line_s1, = ax_a1.plot([], [], color=COLOR_SIATA,  lw=2.5,
                      marker='o', markersize=5, markerfacecolor='white',
                      markeredgewidth=1.8, label='Medición real SIATA', zorder=5)
lbl_b1 = ax_a1.text(0.02, 0.92, '', transform=ax_a1.transAxes,
                    fontsize=10, color=COLOR_BILSTM, fontweight='bold', va='top')
lbl_s1 = ax_a1.text(0.02, 0.84, '', transform=ax_a1.transAxes,
                    fontsize=10, color=COLOR_SIATA, fontweight='bold', va='top')
box_m1 = ax_a1.text(0.97, 0.95, '', transform=ax_a1.transAxes,
                    ha='right', va='top', fontsize=10, fontfamily='monospace',
                    bbox=dict(boxstyle='round,pad=0.5', fc='white',
                              ec=COLOR_BILSTM, alpha=0.0, lw=1.5))

def _update_a1(frame):
    if frame < 26:
        k = max(1, int(frame / 25 * n_pts))
        line_b1.set_data(x_hours[:k], bilstm_vals[:k])
        if frame == 25:
            lbl_b1.set_text('Predicción BiLSTM')
    elif frame < 52:
        line_b1.set_data(x_hours, bilstm_vals)
        lbl_b1.set_text('Predicción BiLSTM')
        k = max(1, int((frame - 26) / 25 * n_pts))
        line_s1.set_data(x_hours[:k], siata_vals[:k])
        if frame == 51:
            lbl_s1.set_text('Medición real SIATA')
    else:
        line_b1.set_data(x_hours, bilstm_vals)
        line_s1.set_data(x_hours, siata_vals)
        lbl_b1.set_text('Predicción BiLSTM')
        lbl_s1.set_text('Medición real SIATA')
        box_m1.set_text(f'RMSE este día: {rmse_day:.1f} W/m²')
        box_m1.get_bbox_patch().set_alpha(0.92)

anim1 = animation.FuncAnimation(
    fig_a1, _update_a1, frames=TOTAL_FRAMES1, interval=80, blit=False)

plt.tight_layout()
# Display as JavaScript animation (no ffmpeg required)
display(HTML(anim1.to_jshtml()))
plt.close(fig_a1)

## Paso 6: ¿Qué hubiera predicho el GFS solo?

El **GFS** (Global Forecast System de la NOAA) es el pronóstico numérico que sirve de entrada al modelo.

Comparamos tres curvas:
1. **GFS sin corrección** — el pronóstico tal como sale del modelo numérico de la NOAA
2. **BiLSTM** — nuestro modelo que corrige el sesgo del GFS con aprendizaje automático
3. **Medición real SIATA** — la referencia de verdad de campo

> El GFS tiende a **sobreestimar** la radiación en el Valle de Aburrá, especialmente en horas de máxima irradiancia, porque no captura la nubosidad local del microclima.

Métricas sobre el **conjunto de prueba** (15% del dataset, nunca visto durante el entrenamiento):

| Modelo | RMSE diurno | SkillScore |
|--------|-------------|------------|
| GFS sin corrección | ~202 W/m² | 0.000 (referencia) |
| **BiLSTM** | **~128 W/m²** | **+0.596** |
| **Mejora** | **−36%** | — |

In [ ]:
# ── Load GFS CSI and convert to GHI for 2021-09-15 ───────────────
try:
    ds_g    = xr.open_dataset(GFS_CSI_FILE, engine='h5netcdf')
    gfs_var = list(ds_g.data_vars)[0]
    t_g_all = pd.to_datetime(ds_g['observation_time'].values)
    v_g_all = ds_g[gfs_var].values.squeeze()
    ds_g.close()

    ds_cs2 = xr.open_dataset(CSI_GHI_FILE, engine='h5netcdf')
    t_cs2  = pd.to_datetime(ds_cs2['observation_time'].values)
    v_cs2  = ds_cs2['clear_sky_ghi'].values.squeeze()
    ds_cs2.close()

    _m_g = (t_g_all >= TARGET_DATE) & (t_g_all < '2021-09-16')
    _m_c = (t_cs2  >= TARGET_DATE) & (t_cs2  < '2021-09-16')
    sr_gfs_csi = pd.Series(v_g_all[_m_g], index=t_g_all[_m_g]).reindex(HOURS_IDX).values
    sr_cs2_day = pd.Series(v_cs2[_m_c],   index=t_cs2[_m_c]).reindex(HOURS_IDX).values
    gfs_vals   = np.nan_to_num(sr_gfs_csi * sr_cs2_day, nan=0.0)
    print(f'GFS cargado: {_m_g.sum()} registros  ·  var: {gfs_var!r}')
except FileNotFoundError:
    print(f'ERROR: No se encontró clearsky_index_GFS_0700.nc:\n  {GFS_CSI_FILE}')
    raise

# Compute metrics
_m3          = (siata_vals > 0) & (gfs_vals > 0) & (bilstm_vals > 0)
rmse_gfs3    = np.sqrt(np.mean((gfs_vals[_m3]    - siata_vals[_m3]) ** 2))
rmse_bilstm3 = np.sqrt(np.mean((bilstm_vals[_m3] - siata_vals[_m3]) ** 2))
mejora_pct   = (rmse_gfs3 - rmse_bilstm3) / rmse_gfs3 * 100
worst_idx    = int(np.argmax(np.abs(gfs_vals - siata_vals)))
print(f'RMSE GFS    : {rmse_gfs3:.1f} W/m²')
print(f'RMSE BiLSTM : {rmse_bilstm3:.1f} W/m²')
print(f'Mejora      : {mejora_pct:.1f}%')

# ── FuncAnimation: 4 phases ───────────────────────────────────────
#   Phase 1 (0–25):  GFS draws
#   Phase 2 (26–51): BiLSTM draws over GFS
#   Phase 3 (52–77): SIATA draws
#   Phase 4 (78–90): all 3 + metrics + annotation on biggest GFS error
TOTAL_FRAMES3 = 91
y_max3 = max(gfs_vals.max(), bilstm_vals.max(), siata_vals.max()) * 1.18

fig_a3, ax_a3 = plt.subplots(figsize=(12, 6))
ax_a3.set_xlim(-0.5, n_pts - 0.5)
ax_a3.set_ylim(0, y_max3)
ax_a3.set_xticks(x_hours)
ax_a3.set_xticklabels(xlabels, rotation=30, ha='right')
ax_a3.set_xlabel('Hora local (Colombia, UTC-5)', fontsize=11)
ax_a3.set_ylabel('GHI (W/m²)', fontsize=11)
ax_a3.set_title('Comparación de modelos — 15 de septiembre 2021', fontsize=14, fontweight='bold')
ax_a3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}'))

line_g3, = ax_a3.plot([], [], color=COLOR_GFS,    lw=2.5, label='GFS sin corrección',    zorder=2)
line_b3, = ax_a3.plot([], [], color=COLOR_BILSTM, lw=2.8, label='BiLSTM (nuestro modelo)', zorder=4)
line_s3, = ax_a3.plot([], [], color=COLOR_SIATA,  lw=2.5,
                      marker='o', markersize=5, markerfacecolor='white',
                      markeredgewidth=1.8, label='Medición real SIATA', zorder=5)
lbl_g3  = ax_a3.text(0.02, 0.92, '', transform=ax_a3.transAxes,
                     fontsize=10, color=COLOR_GFS, fontweight='bold', va='top')
lbl_b3  = ax_a3.text(0.02, 0.84, '', transform=ax_a3.transAxes,
                     fontsize=10, color=COLOR_BILSTM, fontweight='bold', va='top')
lbl_s3  = ax_a3.text(0.02, 0.76, '', transform=ax_a3.transAxes,
                     fontsize=10, color=COLOR_SIATA, fontweight='bold', va='top')
box_m3  = ax_a3.text(0.97, 0.95, '', transform=ax_a3.transAxes,
                     ha='right', va='top', fontsize=9, fontfamily='monospace',
                     bbox=dict(boxstyle='round,pad=0.5', fc='white',
                               ec=COLOR_BILSTM, alpha=0.0, lw=1.5))
# Annotation for biggest GFS error (invisible until phase 4)
_ann3 = ax_a3.annotate(
    '', xy=(worst_idx, float(gfs_vals[worst_idx])),
    xytext=(worst_idx - 1.5, float(gfs_vals[worst_idx]) + 100),
    fontsize=9, color=COLOR_GFS, fontweight='bold', va='bottom',
    arrowprops=dict(arrowstyle='->', color=COLOR_GFS, lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=COLOR_GFS, alpha=0.0))

def _update_a3(frame):
    if frame < 26:
        k = max(1, int(frame / 25 * n_pts))
        line_g3.set_data(x_hours[:k], gfs_vals[:k])
        if frame == 25:
            lbl_g3.set_text('GFS sin corrección')
    elif frame < 52:
        line_g3.set_data(x_hours, gfs_vals)
        lbl_g3.set_text('GFS sin corrección')
        k = max(1, int((frame - 26) / 25 * n_pts))
        line_b3.set_data(x_hours[:k], bilstm_vals[:k])
        if frame == 51:
            lbl_b3.set_text('BiLSTM (nuestro modelo)')
    elif frame < 78:
        line_g3.set_data(x_hours, gfs_vals)
        line_b3.set_data(x_hours, bilstm_vals)
        lbl_g3.set_text('GFS sin corrección')
        lbl_b3.set_text('BiLSTM (nuestro modelo)')
        k = max(1, int((frame - 52) / 25 * n_pts))
        line_s3.set_data(x_hours[:k], siata_vals[:k])
        if frame == 77:
            lbl_s3.set_text('Medición real SIATA')
    else:
        line_g3.set_data(x_hours, gfs_vals)
        line_b3.set_data(x_hours, bilstm_vals)
        line_s3.set_data(x_hours, siata_vals)
        lbl_g3.set_text('GFS sin corrección')
        lbl_b3.set_text('BiLSTM (nuestro modelo)')
        lbl_s3.set_text('Medición real SIATA')
        box_m3.set_text(
            f'RMSE GFS: {rmse_gfs3:.0f} W/m²  |  RMSE BiLSTM: {rmse_bilstm3:.0f} W/m²\n'
            f'Mejora: {mejora_pct:.0f}%')
        box_m3.get_bbox_patch().set_alpha(0.92)
        _ann3.set_text('GFS sobreestima aquí')
        _ann3.get_bbox_patch().set_alpha(0.85)

anim3 = animation.FuncAnimation(
    fig_a3, _update_a3, frames=TOTAL_FRAMES3, interval=80, blit=False)

plt.tight_layout()
display(HTML(anim3.to_jshtml()))
plt.close(fig_a3)

## Paso 7: El producto para el tomador de decisiones

### Flujo operativo diario de Emergente

```
4:00 AM  →  El modelo BiLSTM corre automáticamente
             (usa el pronóstico GFS del lanzamiento 00 UTC)
             ↓
4:05 AM  →  Se genera la curva de producción solar hora por hora
             ↓
4:10 AM  →  Emergente recibe el pronóstico para el día
             ↓
8:00 AM  →  Deadline para declarar la oferta energética a XM
             (mercado mayorista colombiano)
```

### ¿Por qué importa la precisión del pronóstico?

Una mejor predicción reduce el **riesgo de desbalance energético**:
- Si se declara **más** energía de la que se produce → cargo por déficit en XM
- Si se declara **menos** energía de la que se produce → pérdida de ingreso potencial

Un RMSE 36% menor significa declaraciones más exactas → **mayor rentabilidad para la planta solar**.

El gráfico siguiente muestra lo que recibiría Emergente cada mañana:

In [ ]:
# ── Product delivery chart — 2 stacked panels ────────────────────
x_prod    = list(range(len(HOURS_IDX)))
xlabels_p = [ts.strftime('%H:%M') for ts in HOURS_IDX]
bilstm_p  = bilstm_vals.copy()

fig_prod, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [2, 1.2]})

# ── TOP PANEL: hourly bars colored by CSI ────────────────────────
bar_colors_p = []
for csi_v in csi_pred_all:
    if csi_v > 0.7:
        bar_colors_p.append('#F4D35E')   # sunny yellow
    elif csi_v > 0.4:
        bar_colors_p.append('#A8DADC')   # partly cloudy blue
    else:
        bar_colors_p.append('#6C757D')   # cloudy gray

bars = ax_top.bar(x_prod, bilstm_p, color=bar_colors_p,
                  edgecolor='white', linewidth=0.6, zorder=3)

# Value label on top of each bar
for bar, val in zip(bars, bilstm_p):
    if val > 20:
        ax_top.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 10, f'{int(val)}',
                    ha='center', va='bottom', fontsize=9, color='#333333')

ax_top.set_ylabel('GHI predicho (W/m²)', fontsize=11)
ax_top.set_title('Producción solar hora por hora — 15 de septiembre 2021',
                 fontsize=13, fontweight='bold')
ax_top.set_xticks(x_prod)
ax_top.set_xticklabels(xlabels_p, rotation=30, ha='right')
ax_top.set_ylim(0, bilstm_p.max() * 1.22)
ax_top.legend(
    handles=[
        mpatches.Patch(facecolor='#F4D35E', label='Despejado (CSI > 0.7)'),
        mpatches.Patch(facecolor='#A8DADC', label='Parcialmente nublado (CSI 0.4–0.7)'),
        mpatches.Patch(facecolor='#6C757D', label='Nublado (CSI < 0.4)'),
    ],
    loc='upper left', fontsize=9.5, framealpha=0.92)

# ── BOTTOM PANEL: cumulative energy curve ─────────────────────────
cumulative_ghi = np.cumsum(bilstm_p)
total_energy   = cumulative_ghi[-1]

ax_bot.fill_between(x_prod, 0, cumulative_ghi, alpha=0.35, color='#F4D35E', zorder=2)
ax_bot.plot(x_prod, cumulative_ghi, color='#CC8800', lw=2.5, zorder=3)
ax_bot.text(x_prod[-1] - 0.3, cumulative_ghi[-1],
            f'Total: {total_energy:.0f} Wh/m²',
            ha='right', va='bottom', fontsize=11, fontweight='bold', color='#CC8800',
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#CC8800', alpha=0.92))

ax_bot.set_ylabel('Energía acumulada (Wh/m²)', fontsize=11)
ax_bot.set_title('Energía solar acumulada en el día', fontsize=12)
ax_bot.set_xticks(x_prod)
ax_bot.set_xticklabels(xlabels_p, rotation=30, ha='right')
ax_bot.set_ylim(0, total_energy * 1.18)

# ── Footer text box ───────────────────────────────────────────────
fig_prod.text(
    0.5, -0.01,
    'Este pronóstico se entrega a las 4:00 AM y permite a Emergente declarar\n'
    'su oferta energética antes de las 8:00 AM en el mercado mayorista colombiano — XM',
    ha='center', va='top', fontsize=10, style='italic', color='#555555',
    bbox=dict(boxstyle='round,pad=0.5', fc='#F8F9FA', ec='#CCCCCC', alpha=0.9))

fig_prod.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(os.getcwd(), 'graph15_product.png'), dpi=120, bbox_inches='tight')
plt.show()